# Étape 5 : Entraînement du Modèle (Model Training)

## 🎯 Objectif

Construire et **comprendre** un modèle de filtrage collaboratif de base en expérimentant systématiquement avec différents hyperparamètres.

## 📚 Documentation Référence

- **LightFM Model Class** : `5.Documentations/LightFM/LightFM — LightFM 1.16 documentation.pdf`

## 🔬 Configuration Expérimentale

Selon le projet (RecSys Project.docx), nous allons :

1. **Commencer simple** : Modèle utilisant **uniquement les interactions** (sans features)
2. **Tester 3 loss functions** : WARP, BPR, logistic
3. **Expérimenter avec différents nombres de facteurs latents** : 30, 50, 100
4. **Tester différents learning rates** : 0.01, 0.05, 0.1
5. **📊 SURVEILLER LA CONVERGENCE** : Évaluer epoch par epoch

## 📋 Paramètres à Explorer

| Paramètre | Valeurs à Tester | Notes |
|-----------|------------------|-------|
| `no_components` | 30, 50, 100 | Dimensionnalité des embeddings |
| `loss` | 'warp', 'bpr', 'logistic' | Fonction de perte |
| `learning_rate` | 0.01, 0.05, 0.1 | Taux d'apprentissage |
| `epochs` | 10-20 | Surveiller la convergence |

## 🔑 Loss Functions (Implicit Feedback)

Notre dataset H&M contient uniquement des **achats** (interactions positives), pas de ratings explicites.

### 1. **WARP** (Weighted Approximate-Rank Pairwise)
- **Optimise** : **Precision@K** (top de la liste)
- **Principe** : Maximise le rang des exemples positifs par échantillonnage de négatifs
- **Recommandé pour** : Optimiser les top-K recommandations

### 2. **BPR** (Bayesian Personalised Ranking)
- **Optimise** : **ROC AUC** (ranking global)
- **Principe** : Maximise la différence entre positif et négatif aléatoire
- **Recommandé pour** : Bon ranking général

### 3. **Logistic**
- **Optimise** : Log-loss
- **Principe** : Régression logistique classique
- **Utilisé quand** : On a des interactions positives (1) ET négatives (-1)

## 🎓 Méthodologie

Pour chaque configuration, nous allons :

1. **Entraîner epoch par epoch** avec `fit_partial()`
2. **Évaluer après chaque epoch** (Precision@K, Recall@K, AUC)
3. **Tracer les courbes de convergence**
4. **Comparer les configurations**
5. **Identifier la meilleure configuration**

## 1. Configuration et Imports

In [4]:
# Imports standards
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import json
import pickle
import os
import time
from collections import defaultdict

# Imports scipy
from scipy.sparse import load_npz, csr_matrix

# Imports LightFM
try:
    from lightfm import LightFM
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    LIGHTFM_AVAILABLE = True
    print("✅ LightFM installé et disponible")
except ImportError:
    LIGHTFM_AVAILABLE = False
    print("⚠️  LightFM n'est pas installé.")
    raise ImportError("LightFM est requis pour ce notebook")

# Configuration
warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

# Style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Seed pour reproductibilité
np.random.seed(42)

print("✅ Configuration terminée")

✅ LightFM installé et disponible
✅ Configuration terminée


## 2. Chargement des Données (Step 4)

Nous chargeons les matrices train/test créées dans **Step 4** (split temporel).

In [5]:
print("=" * 80)
print("CHARGEMENT DES DONNÉES (STEP 4)")
print("=" * 80)

PROCESSED_DATA_PATH = 'data/processed/'

print("\n📂 Chargement des matrices d'interactions...")

# Charger les matrices train/test
train_interactions = load_npz(PROCESSED_DATA_PATH + 'train_interactions.npz')
test_interactions = load_npz(PROCESSED_DATA_PATH + 'test_interactions.npz')

print(f"   ✓ Train: {train_interactions.shape} - {train_interactions.nnz:,} interactions")
print(f"   ✓ Test: {test_interactions.shape} - {test_interactions.nnz:,} interactions")

# Charger les métadonnées du split
with open(PROCESSED_DATA_PATH + 'split_metadata.json', 'r') as f:
    split_metadata = json.load(f)

print(f"\n📊 Informations du split:")
print(f"   Stratégie: {split_metadata['chosen_strategy']}")
print(f"   Date de split: {split_metadata['split_date']}")
print(f"   Train ratio: {split_metadata['train_ratio']*100:.0f}%")
print(f"   Cutoff date: {split_metadata['cutoff_date']}")

num_users, num_items = train_interactions.shape

print(f"\n📊 DATASET:")
print(f"   Users: {num_users:,}")
print(f"   Items: {num_items:,}")
print(f"   Train interactions: {train_interactions.nnz:,}")
print(f"   Test interactions: {test_interactions.nnz:,}")

print("\n✅ Données chargées")

CHARGEMENT DES DONNÉES (STEP 4)

📂 Chargement des matrices d'interactions...
   ✓ Train: (435005, 29675) - 5,875,495 interactions
   ✓ Test: (435005, 29675) - 498,557 interactions

📊 Informations du split:
   Stratégie: Temporal Split
   Date de split: 2025-10-22 10:49:57
   Train ratio: 90%
   Cutoff date: 2020-08-31

📊 DATASET:
   Users: 435,005
   Items: 29,675
   Train interactions: 5,875,495
   Test interactions: 498,557

✅ Données chargées


## 3. Fonction d'Entraînement avec Monitoring de Convergence

### 🔑 Stratégie

Pour surveiller la convergence, nous utilisons `fit_partial()` de LightFM qui permet d'entraîner **epoch par epoch** et de continuer depuis l'état courant du modèle.

**Approche** :
1. Créer le modèle
2. Pour chaque epoch :
   - Entraîner 1 epoch avec `fit_partial(epochs=1)`
   - Évaluer sur train ET test
   - Sauvegarder les métriques
3. Retourner l'historique complet

In [6]:
def train_and_monitor(model, train_inter, test_inter, n_epochs=10, k=10, verbose=True, sample_users=None):
    """
    Entraîne un modèle LightFM epoch par epoch et surveille la convergence
    
    OPTIMISÉ pour le dataset de 50K:
    - Évalue sur TOUS les users par défaut (dataset petit)
    - 10 epochs par défaut
    
    Args:
        model: LightFM model instance
        train_inter: scipy sparse matrix (train interactions)
        test_inter: scipy sparse matrix (test interactions)
        n_epochs: nombre d'epochs
        k: K pour precision@k et recall@k
        verbose: afficher progression
        sample_users: nombre d'users pour évaluation (None = tous, recommandé)
    
    Returns:
        dict avec historique des métriques
    """
    import time
    from lightfm.evaluation import precision_at_k, recall_at_k, auc_score
    
    history = {
        'epoch': [],
        'train_precision': [],
        'train_recall': [],
        'train_auc': [],
        'test_precision': [],
        'test_recall': [],
        'test_auc': [],
        'epoch_time': []
    }
    
    num_users = train_inter.shape[0]
    
    if sample_users and sample_users < num_users:
        if verbose:
            print(f"\n⚡ Évaluation sur échantillon de {sample_users:,} users (sur {num_users:,})")
    else:
        if verbose:
            print(f"\n📊 Évaluation sur TOUS les {num_users:,} users")
    
    if verbose:
        print(f"\n🔄 Entraînement sur {n_epochs} epochs...")
        print(f"{'Epoch':<6} | {'Train P@{}'.format(k):<10} | {'Test P@{}'.format(k):<10} | {'Train AUC':<10} | {'Test AUC':<10} | {'Time':<8}")
        print("-" * 80)
    
    for epoch in range(1, n_epochs + 1):
        epoch_start = time.time()
        
        # Entraîner 1 epoch
        model.fit_partial(
            interactions=train_inter,
            epochs=1,
            num_threads=4,
            verbose=False
        )
        
        # Évaluer sur train
        train_prec = precision_at_k(model, train_inter, k=k, train_interactions=None, num_threads=4).mean()
        train_rec = recall_at_k(model, train_inter, k=k, train_interactions=None, num_threads=4).mean()
        train_auc = auc_score(model, train_inter, num_threads=4).mean()
        
        # Évaluer sur test
        test_prec = precision_at_k(model, test_inter, k=k, train_interactions=train_inter, num_threads=4).mean()
        test_rec = recall_at_k(model, test_inter, k=k, train_interactions=train_inter, num_threads=4).mean()
        test_auc = auc_score(model, test_inter, train_interactions=train_inter, num_threads=4).mean()
        
        epoch_time = time.time() - epoch_start
        
        # Sauvegarder dans historique
        history['epoch'].append(epoch)
        history['train_precision'].append(train_prec)
        history['train_recall'].append(train_rec)
        history['train_auc'].append(train_auc)
        history['test_precision'].append(test_prec)
        history['test_recall'].append(test_rec)
        history['test_auc'].append(test_auc)
        history['epoch_time'].append(epoch_time)
        
        if verbose:
            print(f"{epoch:<6} | {train_prec:>10.4f} | {test_prec:>10.4f} | {train_auc:>10.4f} | {test_auc:>10.4f} | {epoch_time:>6.2f}s")
    
    if verbose:
        total_time = sum(history['epoch_time'])
        print(f"\n✅ Entraînement terminé en {total_time:.1f}s (moyenne: {total_time/n_epochs:.1f}s/epoch)")
    
    return history

print("✅ Fonction train_and_monitor (OPTIMISÉE pour 50K) définie")
print("   📊 Évaluation sur TOUS les users par défaut")
print("   ⚡ 10 epochs par défaut")

✅ Fonction train_and_monitor définie


## 4. Expérimentation 1 : Comparaison des Loss Functions

### 🧪 Objectif

Comparer les **3 loss functions** pour implicit feedback :
- **WARP** : Optimise Precision@K
- **BPR** : Optimise ROC AUC
- **Logistic** : Log-loss classique

### ⚙️ Configuration Fixe

Pour isoler l'effet de la loss function :
- `no_components=30` (baseline)
- `learning_rate=0.05` (valeur moyenne)
- `epochs=20`
- `k=10` (pour les métriques)

In [ ]:
print("=" * 80)print("EXPÉRIMENTATION 1: COMPARAISON LOSS FUNCTIONS")print("=" * 80)# Configuration fixeNO_COMPONENTS = 30LEARNING_RATE = 0.05N_EPOCHS = 10K = 10print(f"\n⚙️  Configuration fixe:")print(f"   no_components: {NO_COMPONENTS}")print(f"   learning_rate: {LEARNING_RATE}")print(f"   epochs: {N_EPOCHS}")print(f"   k: {K}")# Stocker les résultatsexp1_results = {}# Test des 3 loss functionsloss_functions = ['warp', 'bpr', 'logistic']for loss_fn in loss_functions:    print(f"\n{'='*80}")    print(f"LOSS FUNCTION: {loss_fn.upper()}")    print(f"{'='*80}")        # Créer le modèle    model = LightFM(        loss=loss_fn,        no_components=NO_COMPONENTS,        learning_rate=LEARNING_RATE,        random_state=42    )        # Entraîner avec monitoring    history = train_and_monitor(        model=model,        train_inter=train_interactions,        test_inter=test_interactions,        n_epochs=N_EPOCHS,        k=K,        verbose=True    )        # Sauvegarder les résultats    exp1_results[loss_fn] = {        'model': model,        'history': history,        'final_test_precision': history['test_precision'][-1],        'final_test_recall': history['test_recall'][-1],        'final_test_auc': history['test_auc'][-1]    }print(f"\n{'='*80}")print("RÉSUMÉ EXPÉRIMENTATION 1")print(f"{'='*80}")summary_df = pd.DataFrame({    'Loss': loss_functions,    f'Test Precision@{K}': [exp1_results[loss]['final_test_precision'] for loss in loss_functions],    f'Test Recall@{K}': [exp1_results[loss]['final_test_recall'] for loss in loss_functions],    'Test AUC': [exp1_results[loss]['final_test_auc'] for loss in loss_functions]})print("\n")print(summary_df.to_string(index=False))# Identifier le meilleurbest_loss = summary_df.loc[summary_df[f'Test Precision@{K}'].idxmax(), 'Loss']print(f"\n🏆 Meilleur loss function: {best_loss.upper()}")print(f"   (basé sur Test Precision@{K})")

EXPÉRIMENTATION 1: COMPARAISON LOSS FUNCTIONS

⚙️  Configuration fixe:
   no_components: 30
   learning_rate: 0.05
   epochs: 20
   k: 10

LOSS FUNCTION: WARP

🔄 Entraînement sur 20 epochs...
Epoch  | Train P@10 | Test P@10  | Train AUC  | Test AUC   | Time    
--------------------------------------------------------------------------------


### 4.1 Visualisation des Courbes de Convergence

In [ ]:
# Visualiser les courbes de convergence
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

colors = {'warp': 'darkgreen', 'bpr': 'steelblue', 'logistic': 'coral'}

# 1. Test Precision@K
for loss_fn in loss_functions:
    history = exp1_results[loss_fn]['history']
    axes[0, 0].plot(history['epoch'], history['test_precision'], 
                    marker='o', label=loss_fn.upper(), color=colors[loss_fn], linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=11)
axes[0, 0].set_ylabel(f'Test Precision@{K}', fontsize=11)
axes[0, 0].set_title(f'Convergence: Test Precision@{K}', fontweight='bold', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Test Recall@K
for loss_fn in loss_functions:
    history = exp1_results[loss_fn]['history']
    axes[0, 1].plot(history['epoch'], history['test_recall'], 
                    marker='o', label=loss_fn.upper(), color=colors[loss_fn], linewidth=2)
axes[0, 1].set_xlabel('Epoch', fontsize=11)
axes[0, 1].set_ylabel(f'Test Recall@{K}', fontsize=11)
axes[0, 1].set_title(f'Convergence: Test Recall@{K}', fontweight='bold', fontsize=12)
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Test AUC
for loss_fn in loss_functions:
    history = exp1_results[loss_fn]['history']
    axes[1, 0].plot(history['epoch'], history['test_auc'], 
                    marker='o', label=loss_fn.upper(), color=colors[loss_fn], linewidth=2)
axes[1, 0].set_xlabel('Epoch', fontsize=11)
axes[1, 0].set_ylabel('Test AUC', fontsize=11)
axes[1, 0].set_title('Convergence: Test AUC', fontweight='bold', fontsize=12)
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Train vs Test (WARP seulement pour clarté)
history_warp = exp1_results['warp']['history']
axes[1, 1].plot(history_warp['epoch'], history_warp['train_precision'], 
                marker='o', label='Train', color='steelblue', linewidth=2)
axes[1, 1].plot(history_warp['epoch'], history_warp['test_precision'], 
                marker='s', label='Test', color='coral', linewidth=2)
axes[1, 1].set_xlabel('Epoch', fontsize=11)
axes[1, 1].set_ylabel(f'Precision@{K}', fontsize=11)
axes[1, 1].set_title(f'Train vs Test (WARP) - Overfitting Check', fontweight='bold', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("📊 Visualisations générées")

## 5. Expérimentation 2 : Impact de no_components (Latent Factors)

### 🧪 Objectif

Évaluer l'impact de la **dimensionnalité des embeddings** sur les performances.

### ⚙️ Configuration

- **no_components** : 30, 50, 100 (à tester)
- `loss='warp'` (meilleur de exp1)
- `learning_rate=0.05`
- `epochs=20`

### 💡 Hypothèse

- **Plus de components** → Modèle plus expressif → Meilleures performances
- **Mais** : Risque d'overfitting et temps d'entraînement plus long

In [ ]:
print("=" * 80)print("EXPÉRIMENTATION 2: IMPACT DE NO_COMPONENTS")print("=" * 80)# ConfigurationCOMPONENTS_LIST = [30, 50, 100]LOSS = 'warp'  # Meilleur de exp1LEARNING_RATE = 0.05N_EPOCHS = 10print(f"\n⚙️  Configuration:")print(f"   no_components: {COMPONENTS_LIST}")print(f"   loss: {LOSS}")print(f"   learning_rate: {LEARNING_RATE}")print(f"   epochs: {N_EPOCHS}")# Stocker les résultatsexp2_results = {}for n_comp in COMPONENTS_LIST:    print(f"\n{'='*80}")    print(f"NO_COMPONENTS: {n_comp}")    print(f"{'='*80}")        # Créer le modèle    model = LightFM(        loss=LOSS,        no_components=n_comp,        learning_rate=LEARNING_RATE,        random_state=42    )        # Entraîner avec monitoring    history = train_and_monitor(        model=model,        train_inter=train_interactions,        test_inter=test_interactions,        n_epochs=N_EPOCHS,        k=K,        verbose=True    )        # Sauvegarder    exp2_results[n_comp] = {        'model': model,        'history': history,        'final_test_precision': history['test_precision'][-1],        'final_test_auc': history['test_auc'][-1],        'total_time': sum(history['epoch_time'])    }print(f"\n{'='*80}")print("RÉSUMÉ EXPÉRIMENTATION 2")print(f"{'='*80}")summary_df2 = pd.DataFrame({    'no_components': COMPONENTS_LIST,    f'Test Precision@{K}': [exp2_results[n]['final_test_precision'] for n in COMPONENTS_LIST],    'Test AUC': [exp2_results[n]['final_test_auc'] for n in COMPONENTS_LIST],    'Total Time (s)': [exp2_results[n]['total_time'] for n in COMPONENTS_LIST]})print("\n")print(summary_df2.to_string(index=False))# Meilleur compromis performance/tempsbest_comp = summary_df2.loc[summary_df2[f'Test Precision@{K}'].idxmax(), 'no_components']print(f"\n🏆 Meilleur no_components: {best_comp}")print(f"   (basé sur Test Precision@{K})")

In [ ]:
# Visualiser l'impact de no_components
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Courbes de convergence Precision@K
for n_comp in COMPONENTS_LIST:
    history = exp2_results[n_comp]['history']
    axes[0].plot(history['epoch'], history['test_precision'], 
                marker='o', label=f'{n_comp} components', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel(f'Test Precision@{K}', fontsize=11)
axes[0].set_title(f'Convergence par no_components', fontweight='bold', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Performance finale vs no_components
final_prec = [exp2_results[n]['final_test_precision'] for n in COMPONENTS_LIST]
axes[1].bar(range(len(COMPONENTS_LIST)), final_prec, 
           tick_label=[str(n) for n in COMPONENTS_LIST], 
           color=['steelblue', 'coral', 'darkgreen'], alpha=0.8, edgecolor='black')
axes[1].set_xlabel('no_components', fontsize=11)
axes[1].set_ylabel(f'Test Precision@{K}', fontsize=11)
axes[1].set_title('Performance Finale', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
# Ajouter valeurs
for i, val in enumerate(final_prec):
    axes[1].text(i, val + 0.001, f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

# 3. Temps d'entraînement vs no_components
total_times = [exp2_results[n]['total_time'] for n in COMPONENTS_LIST]
axes[2].bar(range(len(COMPONENTS_LIST)), total_times, 
           tick_label=[str(n) for n in COMPONENTS_LIST], 
           color='orange', alpha=0.8, edgecolor='black')
axes[2].set_xlabel('no_components', fontsize=11)
axes[2].set_ylabel('Temps Total (s)', fontsize=11)
axes[2].set_title('Temps d\'Entraînement', fontweight='bold', fontsize=12)
axes[2].grid(True, alpha=0.3, axis='y')
# Ajouter valeurs
for i, val in enumerate(total_times):
    axes[2].text(i, val + 1, f'{val:.1f}s', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 Visualisations générées")

## 6. Expérimentation 3 : Impact du Learning Rate

### 🧪 Objectif

Tester l'impact du **taux d'apprentissage** sur la vitesse de convergence et les performances finales.

### ⚙️ Configuration

- **learning_rate** : 0.01, 0.05, 0.1 (à tester)
- `loss='warp'`
- `no_components=50` (bon compromis de exp2)
- `epochs=20`

### 💡 Hypothèse

- **Learning rate élevé** → Convergence rapide mais risque d'instabilité
- **Learning rate faible** → Convergence lente mais plus stable

In [ ]:
print("=" * 80)print("EXPÉRIMENTATION 3: IMPACT DU LEARNING RATE")print("=" * 80)# ConfigurationLR_LIST = [0.01, 0.05, 0.1]LOSS = 'warp'NO_COMPONENTS = 50  # Compromis de exp2N_EPOCHS = 10print(f"\n⚙️  Configuration:")print(f"   learning_rate: {LR_LIST}")print(f"   loss: {LOSS}")print(f"   no_components: {NO_COMPONENTS}")print(f"   epochs: {N_EPOCHS}")# Stocker les résultatsexp3_results = {}for lr in LR_LIST:    print(f"\n{'='*80}")    print(f"LEARNING_RATE: {lr}")    print(f"{'='*80}")        # Créer le modèle    model = LightFM(        loss=LOSS,        no_components=NO_COMPONENTS,        learning_rate=lr,        random_state=42    )        # Entraîner avec monitoring    history = train_and_monitor(        model=model,        train_inter=train_interactions,        test_inter=test_interactions,        n_epochs=N_EPOCHS,        k=K,        verbose=True    )        # Sauvegarder    exp3_results[lr] = {        'model': model,        'history': history,        'final_test_precision': history['test_precision'][-1],        'final_test_auc': history['test_auc'][-1]    }print(f"\n{'='*80}")print("RÉSUMÉ EXPÉRIMENTATION 3")print(f"{'='*80}")summary_df3 = pd.DataFrame({    'learning_rate': LR_LIST,    f'Test Precision@{K}': [exp3_results[lr]['final_test_precision'] for lr in LR_LIST],    'Test AUC': [exp3_results[lr]['final_test_auc'] for lr in LR_LIST]})print("\n")print(summary_df3.to_string(index=False))best_lr = summary_df3.loc[summary_df3[f'Test Precision@{K}'].idxmax(), 'learning_rate']print(f"\n🏆 Meilleur learning_rate: {best_lr}")print(f"   (basé sur Test Precision@{K})")

In [ ]:
# Visualiser l'impact du learning rate
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

colors_lr = {0.01: 'steelblue', 0.05: 'coral', 0.1: 'darkgreen'}

# 1. Courbes de convergence
for lr in LR_LIST:
    history = exp3_results[lr]['history']
    axes[0].plot(history['epoch'], history['test_precision'], 
                marker='o', label=f'LR={lr}', color=colors_lr[lr], linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel(f'Test Precision@{K}', fontsize=11)
axes[0].set_title(f'Convergence par Learning Rate', fontweight='bold', fontsize=12)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 2. Performance finale
final_prec_lr = [exp3_results[lr]['final_test_precision'] for lr in LR_LIST]
axes[1].bar(range(len(LR_LIST)), final_prec_lr, 
           tick_label=[str(lr) for lr in LR_LIST], 
           color=['steelblue', 'coral', 'darkgreen'], alpha=0.8, edgecolor='black')
axes[1].set_xlabel('learning_rate', fontsize=11)
axes[1].set_ylabel(f'Test Precision@{K}', fontsize=11)
axes[1].set_title('Performance Finale', fontweight='bold', fontsize=12)
axes[1].grid(True, alpha=0.3, axis='y')
# Ajouter valeurs
for i, val in enumerate(final_prec_lr):
    axes[1].text(i, val + 0.001, f'{val:.4f}', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

print("📊 Visualisations générées")

## 7. Synthèse et Recommandations

### 🎯 Objectif

Consolider les résultats des 3 expérimentations et **identifier la meilleure configuration**.

In [ ]:
print("=" * 80)
print("SYNTHÈSE FINALE - STEP 5")
print("=" * 80)

print("\n📊 RÉSULTATS DES 3 EXPÉRIMENTATIONS:\n")

print("1️⃣  LOSS FUNCTIONS (no_components=30, lr=0.05):")
for loss_fn in ['warp', 'bpr', 'logistic']:
    prec = exp1_results[loss_fn]['final_test_precision']
    print(f"   • {loss_fn.upper():<10} : Precision@{K} = {prec:.4f}")
best_loss_final = max(exp1_results.items(), key=lambda x: x[1]['final_test_precision'])[0]
print(f"   🏆 Meilleur: {best_loss_final.upper()}")

print("\n2️⃣  NO_COMPONENTS (loss=warp, lr=0.05):")
for n_comp in [30, 50, 100]:
    prec = exp2_results[n_comp]['final_test_precision']
    time_taken = exp2_results[n_comp]['total_time']
    print(f"   • {n_comp:>3} components : Precision@{K} = {prec:.4f} | Time = {time_taken:.1f}s")
best_comp_final = max(exp2_results.items(), key=lambda x: x[1]['final_test_precision'])[0]
print(f"   🏆 Meilleur: {best_comp_final} components")

print("\n3️⃣  LEARNING_RATE (loss=warp, no_components=50):")
for lr in [0.01, 0.05, 0.1]:
    prec = exp3_results[lr]['final_test_precision']
    print(f"   • LR={lr:<4} : Precision@{K} = {prec:.4f}")
best_lr_final = max(exp3_results.items(), key=lambda x: x[1]['final_test_precision'])[0]
print(f"   🏆 Meilleur: LR={best_lr_final}")

print("\n" + "=" * 80)
print("🏆 CONFIGURATION OPTIMALE RECOMMANDÉE")
print("=" * 80)

print(f"\n✅ Meilleure configuration pour Collaborative Filtering pur:\n")
print(f"   • loss             : '{best_loss_final}'")
print(f"   • no_components    : {best_comp_final}")
print(f"   • learning_rate    : {best_lr_final}")
print(f"   • epochs           : 20 (convergence atteinte)")

# Estimer la performance attendue
print(f"\n📈 Performance Attendue (Test Set):\n")
print(f"   • Precision@{K}  : ~{exp2_results[best_comp_final]['final_test_precision']:.4f}")
print(f"   • Recall@{K}     : ~{exp2_results[best_comp_final]['history']['test_recall'][-1]:.4f}")
print(f"   • AUC            : ~{exp2_results[best_comp_final]['final_test_auc']:.4f}")

print("\n💡 INSIGHTS CLÉS:\n")
print(f"   1. {best_loss_final.upper()} est le meilleur loss pour optimiser Precision@K (implicit feedback)")
print(f"   2. {best_comp_final} components offre le meilleur compromis performance/temps")
print(f"   3. Learning rate {best_lr_final} assure une convergence stable")
print(f"   4. La convergence est généralement atteinte après ~15-20 epochs")

print("\n🚀 PROCHAINES ÉTAPES:\n")
print("   → Step 6: Ajouter les features (Hybrid Model)")
print("   → Step 7: Hyperparameter Tuning avancé (Grid Search)")
print("   → Step 8: Analyse Cold-start")

## 8. Sauvegarde des Modèles et Résultats

In [ ]:
print("=" * 80)
print("SAUVEGARDE DES RÉSULTATS")
print("=" * 80)

MODELS_PATH = 'models/'
os.makedirs(MODELS_PATH, exist_ok=True)

# 1. Sauvegarder le meilleur modèle de chaque expérimentation
print("\n💾 Sauvegarde des modèles...")

# Meilleur modèle global (exp2 avec best config)
best_model = exp2_results[best_comp_final]['model']
with open(MODELS_PATH + 'step5_best_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)
print(f"   ✓ step5_best_model.pkl")

# 2. Sauvegarder les résultats des expérimentations
print("\n💾 Sauvegarde des résultats...")

results_summary = {
    'timestamp': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'k': K,
    'best_configuration': {
        'loss': best_loss_final,
        'no_components': int(best_comp_final),
        'learning_rate': float(best_lr_final),
        'epochs': N_EPOCHS
    },
    'best_performance': {
        'test_precision_at_k': float(exp2_results[best_comp_final]['final_test_precision']),
        'test_recall_at_k': float(exp2_results[best_comp_final]['history']['test_recall'][-1]),
        'test_auc': float(exp2_results[best_comp_final]['final_test_auc'])
    },
    'exp1_loss_functions': {
        loss: {
            'test_precision': float(exp1_results[loss]['final_test_precision']),
            'test_auc': float(exp1_results[loss]['final_test_auc'])
        } for loss in ['warp', 'bpr', 'logistic']
    },
    'exp2_no_components': {
        str(n): {
            'test_precision': float(exp2_results[n]['final_test_precision']),
            'test_auc': float(exp2_results[n]['final_test_auc']),
            'total_time': float(exp2_results[n]['total_time'])
        } for n in [30, 50, 100]
    },
    'exp3_learning_rate': {
        str(lr): {
            'test_precision': float(exp3_results[lr]['final_test_precision']),
            'test_auc': float(exp3_results[lr]['final_test_auc'])
        } for lr in [0.01, 0.05, 0.1]
    }
}

with open(MODELS_PATH + 'step5_results.json', 'w') as f:
    json.dump(results_summary, f, indent=2)
print(f"   ✓ step5_results.json")

# 3. Sauvegarder les historiques de convergence
print("\n💾 Sauvegarde des historiques...")

# Sauvegarder l'historique du meilleur modèle
best_history = exp2_results[best_comp_final]['history']
history_df = pd.DataFrame(best_history)
history_df.to_csv(MODELS_PATH + 'step5_best_model_history.csv', index=False)
print(f"   ✓ step5_best_model_history.csv")

print("\n" + "=" * 80)
print("✅ STEP 5 TERMINÉ AVEC SUCCÈS")
print("=" * 80)

print(f"\n📦 Fichiers créés dans {MODELS_PATH}:")
print(f"   • step5_best_model.pkl (modèle optimisé)")
print(f"   • step5_results.json (résultats des 3 expérimentations)")
print(f"   • step5_best_model_history.csv (historique de convergence)")

print(f"\n🎯 Configuration optimale identifiée:")
print(f"   loss={best_loss_final}, no_components={best_comp_final}, learning_rate={best_lr_final}")

print(f"\n🚀 Prochaine étape: Step 6 - Hybrid Model avec Features")